# LLM Evaluation Methodology

> After a model finishes training, judging whether it is genuinely stronger than the previous version requires objective, reproducible scores rather than subjective impressions like "it feels better." Evaluation is the process of converting the subjective question "is this model good?" into quantifiable, reproducible objective scores.
>
> This section covers the evaluation systems used in real papers and industry: how metrics are defined, how open-source evaluation frameworks work, how LLM-as-Judge lets a strong model act as a referee, and how to read results and write compelling comparison reports.

The three core elements are standardized test questions (benchmarks), automated grading (metrics), and a reproducible pipeline (evaluation pipeline). LLM-as-Judge is a recent approach: using a strong model like GPT-4 to judge the quality of other models' responses. In settings such as MT-Bench and Chatbot Arena, strong-model judges show high agreement with human preferences; however, this agreement depends on the task, the judge prompt, the model version, and the answer format. Reference: [MT-Bench / Chatbot Arena](https://arxiv.org/abs/2306.05685).

LLM-as-Judge has its own pitfalls — position bias, length bias, and format bias can all distort scores.

Perplexity measures a model's average uncertainty when predicting the next token on a given piece of text. It is well suited for language modeling and pretraining monitoring, but it does not directly represent chat quality, reasoning ability, or safety.

In [ ]:
import sys

print(f"Python: {sys.version.split()[0]}")

# Check key dependencies used in this section
deps = ["openai", "datasets", "lm_eval"]
for pkg in deps:
    try:
        __import__(pkg)
        print(f"  {pkg}: installed")
    except ImportError:
        print(f"  {pkg}: not installed (pip install {pkg})")


## 1. Evaluation Landscape

### 1.1 Evaluation Evolution Timeline (teaching version)

```
2019-2021         2022-2023            2024-2025
  |                  |                     |
GLUE/SuperGLUE   MMLU/GSM8K         LLM-as-Judge
BERT era         GPT-4 era           Agent era
MCQ-centric      MCQ + generation    Multi-turn + tools + safety
```

### 1.2 Common Evaluation Dimensions in Recent Papers and Industry

| Dimension | Representative Dataset | Metric | Why It Matters |
|:---|:---|:---|:---|
| **Knowledge** | MMLU-Pro, GPQA | acc | Common in top-tier model reports |
| **Math Reasoning** | GSM8K, MATH, AIME 2024 | exact_match | Hard metric for reasoning ability |
| **Code** | HumanEval+, LiveCodeBench, SWE-bench | pass@k | What programmers care about most |
| **Instruction Following** | IFEval, MT-Bench | strict_acc | Determines real-world usability |
| **Dialogue Quality** | AlpacaEval, Chatbot Arena | win_rate/Elo | Strong model acts as referee |
| **Safety** | TruthfulQA, Garak | violation rate | Usually required before deployment |
| **Long Context** | Needle-in-Haystack, RULER | recall | Critical for RAG scenarios |
| **Multilingual** | CMMLU, C-Eval | acc | Don't evaluate only in English |
| **Agent** | SWE-bench, WebArena | success_rate | A rapidly growing direction in recent years |

### 1.3 Common Evaluation Suite for General Models

General-purpose model technical reports typically cover these dimensions, but the exact set depends on the model's positioning. Code models, RAG systems, and medical/legal models should not blindly copy general leaderboards:

```
Basics: MMLU-Pro + GPQA + HellaSwag
Code: HumanEval+ + LiveCodeBench + SWE-bench (Agent scenarios)
Math: GSM8K + MATH + AIME 2024
Dialogue: AlpacaEval 2.0 / Chatbot Arena Elo
Instruction: IFEval + MT-Bench
Safety: TruthfulQA + red teaming
```

**Minimal starter suite** (run these 4 first, then expand): GSM8K -> MMLU -> HumanEval -> IFEval

## 2. Core Evaluation Frameworks & Repos

Below are several evaluation frameworks commonly used in industry and academia. They are organized here by teaching utility and do not represent a strict ranking:

### 2.1 lm-evaluation-harness (EleutherAI) — A commonly used open-source evaluation framework

```bash
git clone https://github.com/EleutherAI/lm-evaluation-harness.git
cd lm-evaluation-harness
pip install -e .
```

- **Status**: One of the most commonly used frameworks for open-source model evaluation; many leaderboards and papers use it or its compatible tasks/metrics. When comparing across papers, still verify prompt, few-shot, parsing, and model versions. Reference: [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
- **Capability**: 200+ datasets, one-line command for API / HF / vLLM evaluation
- **OpenAI-Compatible support**: `local-completions` and `local-chat-completions` model types connect to any compatible API

### 2.2 AlpacaEval — A popular LLM-as-Judge tool

```bash
git clone https://github.com/tatsu-lab/alpaca_eval.git
cd alpaca_eval
pip install -e .
```

- **Status**: One of the commonly used automatic dialogue evaluation tools, with particular emphasis on length-controlled win rate; it does not replace manual review and business evaluation. Reference: [AlpacaEval](https://github.com/tatsu-lab/alpaca_eval), [LC AlpacaEval](https://arxiv.org/abs/2404.04475)
- **Core metrics**: LC Win Rate (Length-Controlled, correcting for length bias), WR (raw win rate)
- **800+ prompts**, comparing your model's answers vs GPT-4/Davinci-003, judged by GPT-4

### 2.3 FastChat (LMSYS) — Same as Chatbot Arena

```bash
git clone https://github.com/lm-sys/FastChat.git
cd FastChat
pip install -e ".[eval]"
```

- **Status**: Chatbot Arena is one of the most influential crowdsourced battle evaluation platforms; FastChat provides MT-Bench and related judge tools
- **Core capability**: MT-Bench (80 multi-turn questions + GPT-4 scoring), Chatbot Arena (1M+ human votes)

### 2.4 DeepEval — CI/CD friendly

```bash
pip install deepeval
```

- **Status**: pytest-style, integrates into CI/CD pipelines
- **Features**: Hallucination detection, answer relevance, faithfulness, G-Eval (chain-of-thought evaluation)

### Framework Selection Guide

| Scenario | Recommended Framework |
|:---|:---|
| **Paper reporting / open model evaluation** | lm-evaluation-harness |
| **Dialogue quality assessment** | AlpacaEval / FastChat MT-Bench |
| **CI/CD continuous evaluation** | DeepEval / Promptfoo |
| **Safety testing** | Garak (NVIDIA) |
| **Agent evaluation** | SWE-bench + WebArena |

**Focus of this Part**: Use lm-evaluation-harness for core evaluation, AlpacaEval for dialogue quality evaluation.

## 3. OpenAI-Compatible API Evaluation in Practice

This is the most practical evaluation approach — you deploy an OpenAI-compatible API service (vLLM, Ollama, DeepSeek, various gateways) and evaluate it directly through the API.

### 3.1 Core Principle

lm-evaluation-harness supports OpenAI-compatible APIs through two model types:

| Model Type | API Endpoint | Supported Tasks |
|:---|:---|:---|
| `local-chat-completions` | `/v1/chat/completions` | Generation tasks (GSM8K, HumanEval, IFEval) |
| `local-completions` | `/v1/completions` | Generation + MCQ (MMLU needs logprobs) |

**Key distinction**: Classic MCQ tasks like MMLU / HellaSwag usually require the token logprob of each option. Many Chat Completions APIs do not expose enough logprobs, so the completions/logits path is commonly used; they can also be reformulated into generative MCQ, but the score cannot be directly compared with the default leaderboard.

### 3.2 Supported Services

Any OpenAI-compatible service works:

```
OpenAI API          -> requires API Key
DeepSeek API        -> OpenAI-compatible format
vLLM deployment     -> http://localhost:8000/v1
Ollama              -> http://localhost:11434/v1
LiteLLM proxy       -> unified proxy for multiple backends
SGLang              -> http://localhost:30000/v1
```

**In one sentence**: If the service can receive `POST /v1/chat/completions`, it can usually run generation tasks; but MCQ/loglikelihood tasks also need completions/logprobs or local logits. Different OpenAI-compatible services do not fully agree on support for logprobs, chat template, stop, and reasoning_content.

GSM8K is a generation task and can be evaluated with the Chat Completions API. Below is the command template — replace `base_url`, `model`, and `token` with your actual configuration:

```bash
# Closed-source API (OpenAI)
lm_eval --model local-chat-completions \
    --model_args model=gpt-4o-mini,base_url=https://api.openai.com/v1/chat/completions,token=$OPENAI_API_KEY,num_concurrent=4,max_retries=3,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 \
    --output_path ./eval_results/gsm8k_openai

# Closed-source API (DeepSeek, OpenAI-compatible format)
lm_eval --model local-chat-completions \
    --model_args model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4,max_retries=3,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 \
    --output_path ./eval_results/gsm8k_deepseek

# Local deployment (vLLM)
lm_eval --model local-chat-completions \
    --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/chat/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 \
    --output_path ./eval_results/gsm8k_vllm

# Local deployment (Ollama)
lm_eval --model local-chat-completions \
    --model_args model=llama3,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 \
    --output_path ./eval_results/gsm8k_ollama
```

Prerequisite: `pip install lm-eval` has been completed.

MMLU is an MCQ task, evaluated by default through loglikelihood (computing each option's probability), which requires the Completions API to expose logprobs. The Chat Completions API will raise `NotImplementedError` on loglikelihood tasks.

```bash
# vLLM's /v1/completions endpoint returns logprobs and can run classic MMLU
lm_eval --model local-completions \
    --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
    --tasks mmlu --batch_size 16 \
    --output_path ./eval_results/mmlu
```

Notes:

- OpenAI's `gpt-4o-mini` does not support `/v1/completions`; you need to use older models like `davinci-002`
- DeepSeek's `/v1/completions` may also be unsupported; use the Chat API for generation tasks
- If you only have the Chat API, you can rewrite MMLU into a generative version (letting the model output A/B/C/D), but the score cannot be directly compared with the default leaderboard

Practical advice: Deploy an open-source model with vLLM, use `local-completions` for MMLU, and `local-chat-completions` for GSM8K — this is the most complete workflow.

### 3.3 Batch Evaluation of Multiple Datasets (Python API)

The CLI is suitable for quick one-off verification, but for systematic multi-dataset evaluation in a notebook, the Python API is more flexible. You can write loops over multiple benchmarks, customize the few-shot count and sampling strategy per dataset, and collect results into a unified DataFrame for downstream analysis.

lm-eval's Python API provides a `simple_evaluate` function with parameters mostly matching the CLI, but returning a Python dictionary for direct processing and visualization in code. Below we use it to batch-run three common benchmarks.

In [ ]:
# lm_eval Python API batch evaluation example
# CLI suits single runs; the Python API suits systematic multi-task evaluation in a notebook

try:
    from lm_eval import simple_evaluate
    HAS_LMEVAL = True
except ImportError:
    HAS_LMEVAL = False
    print("lm_eval not installed; below we show the typical usage and result format of simple_evaluate")
    print("After install it can run for real: pip install lm-eval\n")

if HAS_LMEVAL:
    # Real run example (requires API Key or local model)
    results = simple_evaluate(
        model="local-chat-completions",
        model_args="model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4",
        tasks=["gsm8k", "ifeval"],
        batch_size=8,
        output_path="./eval_results/",
    )
    for task, metrics in results["results"].items():
        print(f"{task}: {metrics}")
else:
    # Example output format (reference magnitude from public technical reports, for structure demo only)
    example_results = {
        "gsm8k": {"exact_match,strict-match": 0.834, "exact_match,flexible-extract": 0.871},
        "mmlu": {"acc,none": 0.743, "acc_norm,none": 0.725},
        "hellaswag": {"acc,none": 0.829, "acc_norm,none": 0.841},
        "ifeval": {"prompt_level_strict_acc,none": 0.687, "inst_level_strict_acc,none": 0.763},
        "humaneval": {"pass@1": 0.689},
    }
    for task, metrics in example_results.items():
        print(f"{task}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")


## 4. LLM-as-Judge: Using a Strong Model as Referee

MCQ and math tasks have ground-truth answers, but "is this dialogue good?" does not — that's where **LLM-as-Judge** comes in.

### 4.1 Core Idea

```
Your model's answer  -->  GPT-4 scores it  -->  Compare vs baseline answer  -->  Win Rate
Baseline answer      -->                    -->                                -->
```

Two mainstream frameworks:
- **AlpacaEval 2.0**: 805 prompts, your model vs GPT-4, judged by GPT-4 Turbo
- **MT-Bench**: 80 multi-turn dialogue questions, GPT-4 scores per dimension (1-10)

### 4.2 Key Concepts

| Metric | Meaning | How to Read |
|:---|:---|:---|
| **Win Rate (WR)** | Proportion of your answers judged better than baseline | GPT-4 as baseline ~= 50% |
| **LC Win Rate** | Length-Controlled WR, correcting for answer length bias | Fairer than WR, avoids "longer wins" |
| **MT-Bench Score** | Average 1-10 rating from GPT-4 | 7+ is good, 8+ is strong |
| **Elo Rating** | Chess-style Elo derived from pairwise battle records | Chatbot Arena standard output |

### 4.3 In Practice: Implementing LLM-as-Judge with the OpenAI SDK

Below we write a fully functional judge — no extra framework needed.

In [ ]:
# Implement LLM-as-Judge directly with the OpenAI SDK
# Core: assemble the question, your model's answer, and the reference answer into a prompt, and let GPT-4 score it

JUDGE_PROMPT = """Please act as an impartial judge and evaluate the quality of the response provided by an AI assistant to the user question displayed below.

Your evaluation should consider the following factors:
1. Helpfulness: Does the response address the user's needs?
2. Accuracy: Is the information factually correct?
3. Relevance: Does the response stay on topic?
4. Depth: Does it provide meaningful detail?
5. Creativity: Is the response well-structured and clear?

Begin your evaluation by providing a short explanation. Be as objective as possible.
After providing your explanation, you must rate the response on a scale of 1 to 10 by strictly following this format:
"[[rating]]", for example: "Rating: [[7]]".

[Question]
{question}

[The Start of Assistant's Answer]
{answer}
[The End of Assistant's Answer]"""


# Evaluation samples (MT-Bench style questions)
eval_samples = [
    {
        "question": "Write a Python function to find the longest common subsequence of two strings.",
        "good_answer": "Here's a Python implementation of LCS using dynamic programming:\n\n```python\ndef lcs(s1: str, s2: str) -> str:\n    m, n = len(s1), len(s2)\n    dp = [[\"\"]] * (n + 1) for _ in range(m + 1)]\n    for i in range(1, m + 1):\n        for j in range(1, n + 1):\n            if s1[i-1] == s2[j-1]:\n                dp[i][j] = dp[i-1][j-1] + s1[i-1]\n            else:\n                dp[i][j] = max(dp[i-1][j], dp[i][j-1], key=len)\n    return dp[m][n]\n```\n\nTime complexity: O(mn), Space: O(mn).",
        "bad_answer": "def lcs(s1, s2):\n    return ''.join(c for c in s1 if c in s2)",
    },
    {
        "question": "Explain the concept of quantum entanglement in simple terms.",
        "good_answer": "Quantum entanglement is when two particles become linked in such a way that the state of one instantly influences the state of the other, no matter how far apart they are. Imagine two magic coins: when you flip them, if one shows heads, the other always shows tails -- even if they're on opposite sides of the universe. Einstein called this 'spooky action at a distance' because it seems to violate the idea that nothing can travel faster than light. Today, entanglement is a proven phenomenon and forms the basis for quantum computing and quantum cryptography.",
        "bad_answer": "Quantum entanglement is when two things are connected. It's like twins who can feel each other's pain. Scientists use it for computers.",
    },
]


def judge_with_gpt(question, answer):
    """Score with GPT-4 (requires the OPENAI_API_KEY environment variable)"""
    try:
        from openai import OpenAI
        client = OpenAI()  # Auto-reads OPENAI_API_KEY
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": JUDGE_PROMPT.format(question=question, answer=answer)
            }],
            temperature=0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Judge unavailable: {e}]"


import re
for i, sample in enumerate(eval_samples):
    print(f"Question {i+1}: {sample['question']}")
    print(f"{'-'*60}")
    for label, answer in [("Good Answer", sample['good_answer']), ("Bad Answer", sample['bad_answer'])]:
        verdict = judge_with_gpt(sample['question'], answer)
        match = re.search(r'\[\[(\d+(?:\.\d+)?)\]\]', verdict)
        if match:
            print(f"  {label}: {match.group(1)}/10")
        else:
            preview = verdict[:150].replace('\n', ' ')
            print(f"  {label}: {preview}...")
    print()

print("At scale: N prompts x M models x GPT-4 judge = a full LLM-as-Judge evaluation")
print("This is the core principle behind MT-Bench and AlpacaEval.")


## 5. Aggregating and Comparing Evaluation Results


In [ ]:
# Evaluation result aggregation and visualization
# Sample data: used to demonstrate the format of aggregation and visualization; a real report must label the source, model version, and eval config for each item
benchmark_results = {
    "GPT-4o": {
        "MMLU": 88.7, "GSM8K": 96.1, "HumanEval": 90.2, "HellaSwag": 95.3,
        "IFEval": 84.3, "GPQA": 53.6, "AlpacaEval LC": 57.5,
    },
    "DeepSeek-V3 (671B)": {
        "MMLU": 88.5, "GSM8K": 95.3, "HumanEval": 82.6, "HellaSwag": 89.0,
        "IFEval": 86.1, "GPQA": 59.1, "AlpacaEval LC": 54.2,
    },
    "Qwen2.5-72B": {
        "MMLU": 86.1, "GSM8K": 91.6, "HumanEval": 86.6, "HellaSwag": 86.9,
        "IFEval": 81.7, "GPQA": 49.0, "AlpacaEval LC": 50.5,
    },
    "Llama-3.1-70B": {
        "MMLU": 86.0, "GSM8K": 91.2, "HumanEval": 80.5, "HellaSwag": 85.0,
        "IFEval": 80.4, "GPQA": 46.7, "AlpacaEval LC": 44.8,
    },
    "Qwen2.5-7B (ref)": {
        "MMLU": 74.3, "GSM8K": 83.4, "HumanEval": 68.9, "HellaSwag": 82.1,
        "IFEval": 68.7, "GPQA": 36.7, "AlpacaEval LC": 38.5,
    },
}

datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA", "AlpacaEval LC"]

# Print comparison table
print("Model Evaluation Comparison Table (sample data, for format demo)\n")

header = f"{'Model':<22s}"
for ds in datasets:
    header += f" {ds:>13s}"
print(header)
print("-" * (22 + 14 * len(datasets)))

for model, scores in benchmark_results.items():
    row = f"{model:<22s}"
    for ds in datasets:
        score = scores.get(ds)
        if score is None:
            row += f" {'N/A':>13s}"
        else:
            row += f" {score:>13.1f}"
    print(row)

# Key analysis
print("\nAnalysis Points\n")
print("1. Compare with same-scale models (7B vs 7B), not across scales (7B vs 671B)")
print("2. Focus on metrics most relevant to your business: API products -> IFEval, education -> MMLU")
print("3. Look at trends across multiple datasets, don't fixate on a single number")
print("4. Label data source and eval config (prompt/few-shot/seed) for reproducibility")
print("5. Model size isn't the only factor; different models are strong on some tasks and weak on others, so conclusions must be tied to a specific benchmark and eval config")


### 5.1 Visualizing Evaluation Results

Number tables alone aren't intuitive. Two chart types commonly used in papers and leaderboards:

| Chart Type | Use Case | Example |
|:---|:---|:---|
| **Radar/Spider Chart** | Multi-dimensional comparison of 2-4 models | Standard in OpenAI/DeepSeek technical reports |
| **Grouped Bar Chart** | Item-by-item comparison of multiple models across multiple datasets | Common in paper ablation studies |
| **Win Rate Matrix** | Pairwise comparison results from LLM-as-Judge | Chatbot Arena style |

In [ ]:
# Radar chart + bar chart + win rate matrix: the three standard visualizations in papers
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# ============================================
# Chart 1: Radar Chart (Spider/Radar Chart)
# ============================================
radar_models = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (ref)"]
radar_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]
radar_colors = ["#2563EB", "#10B981", "#F59E0B", "#EF4444"]

radar_values = []
for model in radar_models:
    vals = [benchmark_results[model][ds] for ds in radar_datasets]
    radar_values.append(vals)

num_vars = len(radar_datasets)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model, values, color in zip(radar_models, radar_values, radar_colors):
    values_closed = values + values[:1]
    ax.fill(angles, values_closed, alpha=0.05, color=color)
    ax.plot(angles, values_closed, "o-", linewidth=2, color=color, label=model, markersize=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_datasets, fontsize=11)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=8, color="gray")
ax.set_title("Model Comparison -- Radar Chart", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Radar chart: larger area = stronger; shape reveals capability distribution")
print("This is a reading of the sample data above and does not represent a live model ranking\n")

# ============================================
# Chart 2: Grouped Bar Chart
# ============================================
bar_models = radar_models
bar_datasets = radar_datasets

x = np.arange(len(bar_datasets))
width = 0.2
n_models = len(bar_models)

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model, color) in enumerate(zip(bar_models, radar_colors)):
    values = [benchmark_results[model][ds] for ds in bar_datasets]
    offset = width * (i - n_models/2 + 0.5)
    bars = ax.bar(x + offset, values, width, label=model, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f"{val:.1f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.set_xlabel("Benchmark", fontsize=12)
ax.set_ylabel("Score (%)", fontsize=12)
ax.set_title("Model Comparison -- Grouped Bar Chart", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(bar_datasets, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print("Bar chart: item-by-item comparison at a glance, good for paper ablations and presentations")
print("The gap between the reference model (red) and the top models on each benchmark is the room for improvement\n")

# ============================================
# Chart 3: Win Rate Matrix (LLM-as-Judge style)
# ============================================
print("--- LLM-as-Judge Win Rate Matrix Example ---\n")

models_for_matrix = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (ref)"]
n_mat = len(models_for_matrix)
# Example win rate matrix: row model's win rate vs column model
win_rate_matrix = np.array([
    [0.50, 0.55, 0.62, 0.85],
    [0.45, 0.50, 0.57, 0.80],
    [0.38, 0.43, 0.50, 0.72],
    [0.15, 0.20, 0.28, 0.50],
])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(win_rate_matrix, cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(n_mat))
ax.set_yticks(range(n_mat))
ax.set_xticklabels(models_for_matrix, fontsize=10, rotation=30, ha="right")
ax.set_yticklabels(models_for_matrix, fontsize=10)
ax.set_title("Pairwise Win Rate Matrix\n(row model vs col model)", fontsize=12, fontweight="bold")

for i in range(n_mat):
    for j in range(n_mat):
        color = "white" if win_rate_matrix[i][j] < 0.3 or win_rate_matrix[i][j] > 0.7 else "black"
        ax.text(j, i, f"{win_rate_matrix[i][j]:.2f}", ha="center", va="center", fontsize=11, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, label="Win Rate")
plt.tight_layout()
plt.show()

print("Win rate matrix: row model vs column model, >0.5 means the row model is stronger")
print("Qwen2.5-7B is <0.5 in all comparisons, showing a clear gap with top models")


In [ ]:
# Composite score calculation: comparing multiple aggregation methods
import numpy as np
from scipy import stats

print("=== Composite Score Calculation ===\n")

# Use the benchmark_results from earlier
scores = benchmark_results

# Select benchmarks for calculation (exclude AlpacaEval LC since the max score is not 100-based)
eval_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]

print("### Method Comparison\n")

# --- 1. Simple Average ---
print("1. Arithmetic Mean")
print(f"{'Model':<22s} {'Avg':>6s} {'Std':>6s}")
print("-" * 36)
for model in scores:
    vals = [scores[model][ds] for ds in eval_datasets]
    avg = np.mean(vals)
    std = np.std(vals)
    print(f"{model:<22s} {avg:>6.1f} {std:>6.1f}")

# --- 2. Geometric Mean ---
print(f"\n2. Geometric Mean -- penalizes weak spots")
print(f"{'Model':<22s} {'GMean':>6s}")
print("-" * 30)
for model in scores:
    vals = [scores[model][ds] for ds in eval_datasets]
    gmean = stats.gmean(vals)
    print(f"{model:<22s} {gmean:>6.1f}")

# --- 3. Weighted Average ---
print(f"\n3. Weighted Average -- assuming knowledge + code carry higher weights")
weights = {"MMLU": 0.25, "GSM8K": 0.20, "HumanEval": 0.20, "HellaSwag": 0.10, "IFEval": 0.15, "GPQA": 0.10}
print(f"   Weights: {weights}")
print(f"{'Model':<22s} {'Weighted':>8s}")
print("-" * 32)
for model in scores:
    weighted = sum(scores[model][ds] * weights[ds] for ds in eval_datasets)
    print(f"{model:<22s} {weighted:>8.1f}")

# --- 4. Normalized to Best Model ---
print(f"\n4. Normalized Average -- GPT-4o as the 100% baseline")
baseline = "GPT-4o"
print(f"{'Model':<22s}", end="")
for ds in eval_datasets:
    print(f" {ds:>8s}", end="")
print(f" {'Avg%':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 8))
for model in scores:
    normalized = [scores[model][ds] / scores[baseline][ds] * 100 for ds in eval_datasets]
    avg_norm = np.mean(normalized)
    print(f"{model:<22s}", end="")
    for nv in normalized:
        print(f" {nv:>8.1f}", end="")
    print(f" {avg_norm:>8.1f}")

# --- 5. Rank Sum ---
print(f"\n5. Rank Sum -- lower is better")
models_list = list(scores.keys())
ranks = {ds: np.argsort([-scores[m][ds] for m in models_list]).argsort() + 1 for ds in eval_datasets}
print(f"{'Model':<22s}", end="")
for ds in eval_datasets:
    print(f" {ds:>8s}", end="")
print(f" {'Sum':>6s} {'AvgRank':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 14))
for i, model in enumerate(models_list):
    rank_list = [ranks[ds][i] for ds in eval_datasets]
    rank_sum = sum(rank_list)
    rank_avg = np.mean(rank_list)
    print(f"{model:<22s}", end="")
    for r in rank_list:
        print(f" {r:>8.0f}", end="")
    print(f" {rank_sum:>6.0f} {rank_avg:>8.1f}")

# --- Summary ---
print(f"\n### Composite Score Recommendations")
print("  For papers: arithmetic mean + geometric mean reported together (shows overall level and weak spots)")
print("  For slides: pick 2-3 methods, attach a radar chart")
print("  For decisions: weighted average (weights = business priorities)")
print("  For leaderboards: rank sum or normalized average (common on the HuggingFace Leaderboard)")
print("  Geometric mean is most sensitive to weak spots -- if your model is unbalanced, it shows immediately")

### 5.2 How to Calculate Composite Scores

Per-dataset scores alone are not enough; papers often need a single "composite capability score." Common methods:

| Method | Formula | Use Case |
|:---|:---|:---|
| **Arithmetic Mean (Avg)** | $\frac{1}{N}\sum s_i$ | Quick comparison, but sensitive to outliers |
| **Normalized Average** | $\frac{1}{N}\sum \frac{s_i}{\max(s_i)}$ | When benchmarks have different scales |
| **Weighted Average** | $\sum w_i s_i$ | When some dimensions matter more (e.g., business relevance) |
| **Geometric Mean** | $(\prod s_i)^{1/N}$ | Penalizes weak spots; good for emphasizing balanced ability |
| **Rank Sum** | $\sum rank_i$ | Only looks at relative ranking, not absolute scores |
| **Elo Rating** | Derived from pairwise comparisons | LLM-as-Judge scenarios |

**Key principles**:
- Don't just look at the average: a model scoring 90 on math but 10 on safety has an average of 50, which looks ordinary, but the safety issue is critical
- **Geometric mean and arithmetic mean emphasize different things**: the geometric mean penalizes weak spots more and is good for emphasizing balanced ability; the arithmetic mean is more intuitive, and the weighted mean is better for business decisions
- Weights are determined by business: customer service should raise safety/instruction-following weights, education should raise knowledge weights

## 6. AlpacaEval in Practice

The previous section implemented the core logic of LLM-as-Judge, but hand-writing prompts and parsing results is fragile. AlpacaEval wraps the complete workflow.

### 6.1 AlpacaEval Workflow

```
1. Load 805 test prompts
2. Generate answers from your model (OpenAI-Compatible API)
3. GPT-4 judges: your answer vs reference answer, which is better
4. Output: Win Rate / LC Win Rate / Avg Length
```

### 6.2 CLI Method

```bash
# Step 1: Generate your model's answers
alpaca_eval evaluate_from_model \
    --model_name_or_path "your-model" \
    --output_path results/your-model \
    --max_instances 100  # Quick test with 100 samples first

# Step 2: Judge with GPT-4
export OPENAI_API_KEY="sk-xxx"
alpaca_eval evaluate \
    --annotators_config "alpaca_eval_gpt4_turbo_fn" \
    --model_outputs "results/your-model.json" \
    --output_path "results/your-model-eval"

# Step 3: View results
cat results/your-model-eval/leaderboard.csv
```

### 6.3 Python API Method (Running in a Notebook)

```python
from alpaca_eval import evaluate
import pandas as pd

df = evaluate(
    model_outputs="results/your-model-outputs.json",
    annotators_config="alpaca_eval_gpt4_turbo_fn",
    max_instances=100,
)
print(f"Win Rate: {df['win_rate'].iloc[0]:.1%}")
print(f"LC Win Rate: {df['lc_win_rate'].iloc[0]:.1%}")
print(f"Avg Length: {df['avg_length'].iloc[0]:.0f} chars")
```

### 6.4 Interpreting AlpacaEval Results

| Metric | Meaning | Reference Value |
|:---|:---|:---|
| **Win Rate** | Proportion of your answers judged better than reference by GPT-4 | Reference value varies with leaderboard, baseline, and annotator model |
| **LC Win Rate** | Win rate after controlling for length bias | Corrects the advantage from length; may be higher or lower than raw WR; also check the average length |
| **Avg Length** | Average length of your answers | If much longer than reference, WR may be inflated |

### 6.5 Running AlpacaEval with an OpenAI-Compatible API

If your model is deployed as an OpenAI-Compatible API, the integration looks like this:

```python
from openai import OpenAI
import json

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
with open("alpaca_eval/prompts/alpaca_eval.json") as f:
    prompts = json.load(f)

outputs = []
for item in prompts[:100]:  # Start with 100 samples
    resp = client.chat.completions.create(
        model="your-model",
        messages=[{"role": "user", "content": item["instruction"]}],
        temperature=0,
        max_tokens=1024,
    )
    outputs.append({
        "instruction": item["instruction"],
        "output": resp.choices[0].message.content,
        "generator": "your-model",
    })

with open("results/your-model-outputs.json", "w") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)
print(f"Generated {len(outputs)} answers. Next: alpaca_eval evaluate ...")
```

## 7. Specialized Evaluation

### 7.1 RAG System Evaluation

If you're building a RAG application, basic benchmarks aren't enough — you need specialized evaluation:

| Stage | Metric | Meaning |
|:---|:---|:---|
| **Retrieval** | Context Precision / Recall | Whether retrieved documents contain the information needed to answer the question |
| **Retrieval** | NDCG / MRR | Whether relevant documents rank near the top |
| **Generation** | Faithfulness | Whether every claim in the answer can be traced to retrieved context |
| **Generation** | Answer Relevance | Whether the answer addresses the user's question |
| **System** | Noise Sensitivity | Whether answer quality degrades when irrelevant documents are retrieved |

**Recommended tool**:

```bash
pip install ragas  # for RAG evaluation
```

**Key insight**: System performance is not the sum of component performance. Good retrieval plus good generation does not necessarily yield good end-to-end performance — the retrieved content may be misused by the model.

### 7.2 Code Model Evaluation: pass@k vs Stability

| Metric | Meaning | Scenario |
|:---|:---|:---|
| **pass@1** | Generate once, test pass rate | Actual usage (user sees only one output) |
| **pass@k** | Generate k times, at least one passes | Optimistic estimate (multiple tries always find the right answer) |
| **repeated pass / all-pass@k (custom)** | Generate k times, all pass | Business stability analysis; do not conflate with the official pass@k |

The HumanEval paper commonly reports pass@k; SWE-bench reports resolved rate. If your business cares about "correct every time," you can additionally track repeated pass rate or all-pass@k, but note that these are custom stability metrics.

## 8. LLM-as-Judge Bias and Consistency

### 8.1 Known Biases in LLM-as-Judge

| Bias Type | Manifestation | Mitigation |
|:---|:---|:---|
| **Position Bias** | Always favors the answer in the first or second position | Randomize positions, average two judgments |
| **Length Bias** | Longer answers score higher | Use LC (Length-Controlled) Win Rate |
| **Egocentric Bias** | The judge may favor answers whose training style, format, or safety policy resemble its own; the specific bias needs experimental verification | Cross-validate with different judge models (GPT-4 + Claude + open-source judge) |
| **Verbosity Bias** | Verbose but seemingly professional answers score higher | Have the judge score per dimension + annotate citations |

**Best practices**:

- Randomize comparison order
- Sample for manual review; the proportion depends on risk, budget, and eval-set size — high-risk business lines should raise the manual review proportion
- Cross-validate with multiple judge models

### 8.2 Consistency and Robustness Evaluation

Accuracy is not the same as reliability. A model may answer 85% correctly, but give a different answer each time for the same question — such a model cannot go to production.

| Metric | Meaning | Production Significance |
|:---|:---|:---|
| **CR@K (Consistency Rate)** | Ask the same question K times, is it correct every time? | Users expect stable answers |
| **Prompt Robustness** | If you rephrase the question, does the answer stay the same? | Users won't follow your prompt template |
| **Order Robustness** | If you shuffle MCQ options, does the answer change? | True bias detection |
| **Sampling Robustness** | If you set temperature to 0.3, is the output still stable? | Production deployment diversity control |

```
High accuracy + low consistency = high production risk, especially in high-risk scenarios like customer service, finance, healthcare, and code execution
Low accuracy + high consistency = optimizable (at least you know where it's unstable)
```

## 9. Evaluation Metrics System

### 9.1 Which Metric for Which Task

| Task Type | Evaluation Method | Metric | Example Datasets |
|:---|:---|:---|:---|
| **MCQ** | `loglikelihood`: compute each option's probability, take the max | `acc` / `acc_norm` | MMLU, HellaSwag, ARC |
| **Generation** | `generate_until`: autoregressive generation, extract the answer | `exact_match` / `pass@k` / `F1` | GSM8K, HumanEval |
| **Dialogue** | LLM-as-Judge: GPT-4 scoring or comparison | `win_rate` / `Elo` / `score` | MT-Bench, AlpacaEval |
| **Instruction Following** | Rule checking: format, keywords, constraints | `strict_acc` | IFEval |
| **Perplexity** | Sum of logprobs over the full sequence | `ppl` / `bpb` | WikiText, Lambada |

### 9.2 Metrics Increasingly Common in Recent Top-Tier Model Reports

| Metric | Meaning | Why New |
|:---|:---|:---|
| **IFEval strict** | Strictly checks whether format, word count, and keyword constraints are followed | Measures "obedience" ability |
| **GPQA** | Google-Proof Q&A, PhD-level science questions | Distinguishes top-tier models |
| **AIME pass@1** | 15 problems from the American Invitational Mathematics Examination, open-ended | New benchmark after GSM8K saturation |
| **SWE-bench** | Real GitHub issue -> fix bug -> run tests | Measures Agent code ability |
| **LiveCodeBench** | Continuously pulls new problems from LeetCode/Codeforces | Prevents data contamination |
| **RULER** | Long-context retrieval (up to 128K tokens) | Upgrade over Needle-in-Haystack |

### 9.3 Metric Selection Decision Tree

```
What does your model primarily do?
|-- Chat dialogue        -> MT-Bench + AlpacaEval 2.0
|-- Write code           -> HumanEval+ + LiveCodeBench + SWE-bench
|-- Solve math problems  -> GSM8K + MATH + AIME 2024
|-- Domain knowledge     -> MMLU-Pro + GPQA
|-- Follow instructions  -> IFEval + MT-Bench (instruction-following dimension)
|-- RAG applications     -> RAGAS (faithfulness + relevance + context precision)
|-- Agent                -> SWE-bench + WebArena + ToolBench
```

## 10. Common Pitfalls and Best Practices

### 10.1 Data-Level

| Pitfall | Severity | Description | How to Avoid |
|:---|:---|:---|:---|
| **Data Contamination** | Critical | Eval questions leaked into training data, inflating scores | Use dynamic datasets like LiveCodeBench; run decontamination checks via exact / n-gram / embedding matching on training data and eval questions. `--check_integrity` mainly checks task and data integrity and cannot replace decontamination. |
| **Prompt Sensitivity** | Critical | Same model, different prompts can differ by 5-15% | Use OLMES standard prompts; record the prompt version |
| **Few-shot Count** | Important | 0-shot/5-shot/8-shot give different results | Align with papers when comparing. Classic papers commonly use MMLU 5-shot, GSM8K 8-shot, but newer reports also use 0-shot, CoT, maj@k, or tool-assisted. |
| **Stale Eval Data** | Important | HumanEval has been memorized by most models | Use HumanEval+ or LiveCodeBench |
| **Dataset Too Small** | Important | HumanEval has only 164 problems, high variance | Cross-validate with multiple datasets |

### 10.2 Engineering-Level

| Pitfall | Description | How to Avoid |
|:---|:---|:---|:---|
| **Confusing Chat vs Completions API** | MMLU errors directly with Chat API | Use Completions API for MCQ, Chat API for generation |
| **Improper batch size** | Too large -> OOM, too small -> slow | vLLM: 8-16, HF: auto |
| **Seed not fixed** | Different results each run | `--model_args seed=42` |
| **Temperature not 0** | Most deterministic evals use temperature=0; but pass@k, self-consistency, creative writing, or robustness evals may need non-zero temperature | `temperature=0` |
| **Too few concurrent requests** | API evaluation is too slow to iterate on | `num_concurrent=4-8` (depending on API rate limits)|

### 10.3 Interpretation-Level

| Pitfall | Description |
|:---|:---|
| **Reporting only the best score** | Should report mean +/- std over multiple runs |
| **Cross-paper comparison** | Different eval configs (prompt/few-shot/parsing) make direct comparison invalid |
| **Only looking at the total score** | MMLU has 57 subjects — check per-subject; some are good, some are bad |
| **Ignoring length bias** | Longer answers score higher with LLM-as-Judge — use LC Win Rate to correct |

## 11. Quick Reference

```bash
# === Quick Start (running in 30 seconds) ===
# Install
pip install lm-eval openai

# Run GSM8K with the DeepSeek API (generation task, Chat API)
lm_eval --model local-chat-completions \
    --model_args model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 --limit 50 \
    --output_path ./eval_results/

# Run MMLU on a vLLM-deployed open model (MCQ, Completions API)
lm_eval --model local-completions \
    --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/completions,num_concurrent=4,tokenized_requests=False \
    --tasks mmlu --batch_size 16 --limit 100 \
    --output_path ./eval_results/

# Load a local model directly from HuggingFace
lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-7B-Instruct,dtype=bfloat16 \
    --tasks gsm8k,mmlu,hellaswag --batch_size auto \
    --output_path ./eval_results/

# === List supported datasets ===
lm_eval --tasks list | head -30

# === Using the Python API (good for Notebooks/scripts) ===
# from lm_eval import simple_evaluate
# results = simple_evaluate(
#     model='local-chat-completions',
#     model_args='model=deepseek-chat,base_url=...,token=...',
#     tasks=['gsm8k', 'ifeval'],
#     limit=50,
# )
```

## Summary

### What You Learned (by section order)

| # | Section | Core Content |
|:---|:---|:---|
| 1 | Evaluation Landscape | What 2025 papers/industry evaluate, evaluation evolution, minimal starter suite |
| 2 | Core Repos | Selection among lm-eval-harness, AlpacaEval, FastChat, DeepEval |
| 3 | OpenAI-Compatible API | `local-chat-completions` vs `local-completions`, connecting any compatible API |
| 4 | LLM-as-Judge | MT-Bench prompt + OpenAI SDK real scoring implementation |
| 5 | Results Aggregation & Visualization | Comparison table + radar chart + bar chart + win rate matrix + 5 composite score methods |
| 6 | AlpacaEval in Practice | Complete CLI + Python API pipeline, OpenAI-Compatible integration |
| 7 | Specialized Evaluation | RAG (RAGAS faithfulness/relevance), Code (pass@k vs stability) |
| 8 | LLM-as-Judge Bias & Consistency | Position/Length/Egocentric Bias + CR@K/Prompt Robustness |
| 9 | Metrics System | acc / exact_match / pass@k / win_rate / Elo, 2025 new metrics (AIME, SWE-bench, LiveCodeBench) |
| 10 | Common Pitfalls | Data level (contamination/prompt sensitivity/few-shot), engineering level (API confusion/seed/temperature), interpretation level |
| 11 | Quick Reference | CLI + Python API, adjustable to your environment before running |

### Recommended Repos

```bash
# Core evaluation frameworks
git clone https://github.com/EleutherAI/lm-evaluation-harness.git  # Common open-source evaluation framework, supports many tasks
git clone https://github.com/tatsu-lab/alpaca_eval.git              # LLM-as-Judge, 805 prompts
git clone https://github.com/lm-sys/FastChat.git                    # MT-Bench + Chatbot Arena

# Advanced tools
pip install deepeval       # CI/CD evaluation (hallucination detection, G-Eval, 40+ metrics)
pip install ragas          # RAG evaluation (faithfulness, relevance)
```

### Next Steps

```
Level 1 (do today):
  1. pip install lm-eval openai
  2. Run gsm8k --limit 50 with the DeepSeek API
  3. Understand every field in the output JSON

Level 2 (this month):
  1. Deploy an open-source model with vLLM
  2. Run 4 core datasets: gsm8k + mmlu + humaneval + ifeval
  3. Draw a radar chart + compute the geometric mean
  4. Verify scores against a public leaderboard

Level 3 (ongoing):
  1. Integrate AlpacaEval / MT-Bench to evaluate dialogue quality
  2. Build custom eval sets for your business scenario (RAGAS / custom prompts)
  3. Track consistency metrics (CR@K), not just accuracy
```

### Key Concepts Quick Reference

| Concept | One-Line Explanation |
|:---|:---|
| **loglikelihood vs generate_until** | MCQ computes probability (Completions API), generation uses autoregression (Chat API) |
| **pass@k and stability** | pass@k is optimistic (at least one of k correct); stability can be tracked with custom repeated pass / all-pass@k |
| **LC Win Rate** | Win rate after controlling for answer length, avoiding "longer wins" |
| **Geometric vs Arithmetic Mean** | Geometric mean penalizes weak spots; model imbalance shows immediately |
| **CR@K** | Ask the same question K times, the proportion of correct answers — measures stability |
| **Position / Length / Egocentric Bias** | The three major LLM-as-Judge biases, requiring randomization + LC + cross-validation |
| **Data Contamination** | Training set leaked eval questions causing inflated scores — use dynamic datasets like LiveCodeBench to prevent it |

Bottom line: deploying a model without evaluation carries high risk. Evaluation = standardized test questions + automated grading + reproducible scores + continuous monitoring.

## Exercises

1. **Perplexity by Hand**

   Given a 3-token sequence, the model's output logit probabilities (after softmax) are [0.5, 0.3, 0.2], [0.1, 0.7, 0.2], [0.4, 0.1, 0.5]. Calculate perplexity by hand.

   <details><summary>Hint</summary>Take the log probability for each token, average them, then exponentiate. Lower perplexity means the model is more "confident."</details>

2. **LLM-as-Judge Bias Detection**

   Design an experiment: for the same pair of answers, swap positions (answer_a and answer_b), repeat 3 times, and tally the difference in GPT-4's scores. Compute the Position Bias rate in code.

   <details><summary>Hint</summary>Position Bias rate = number of inconsistent scores before/after swapping / total count. Inconsistency means the judge was influenced by position.</details>

3. **Evaluation Report Writing**

   Using the simulated data below, draw a radar chart, compute the geometric mean score of both models across 4 benchmarks, and write your conclusions.

   ```python
   scores = {
       'Model A': {'gsm8k': 82, 'mmlu': 71, 'humaneval': 65, 'ifeval': 78},
       'Model B': {'gsm8k': 78, 'mmlu': 75, 'humaneval': 70, 'ifeval': 72},
   }
   ```

   <details><summary>Hint</summary>Geometric mean = (a * b * c * d) ** (1/4). Compare both models' geometric means and observe which model is more "balanced" on the radar chart.</details>